# Day 59 Practice: Reliability and Responsible AI

Small examples of each technique, then apply them to **your own** capstone during build time.

✅ Exercises 1–3: everyone (about 20 min) · Exercise 4: most of you · Exercise 5: everyone, it is your own
README and demo script · Exercise 6: challenge

In [ ]:
from pathlib import Path

import pandas as pd

## Exercise 1: A failure-case table

`average_from_text` was written for a feedback app: the user types ratings separated by commas.
`try_input` (given) runs a function on one input and reports the result or the **name** of the error.

1. Run the table as given and fill in the *What happens now* column below.
2. Decide *What should happen* for each case.
3. Write `safe_average(text)`: ignore empty parts (`"4,,5"`), accept only whole numbers 1–5, and return a dict
   `{"average": ..., "skipped": [...]}` with the parts it could not use. If **no** valid rating is left, raise
   `ValueError("No valid ratings 1-5 found")`.

| Input | What happens now | What should happen |
|-------|------------------|--------------------|
| `"4, 5, 3"` | | |
| `""` | | |
| `"4, five"` | | |
| `"4,,5"` | | |
| `"9, 1"` | | |
| `"4.5, 3"` | | |

Expected output of the fixed version:
```
'4, 5, 3'  -> ok: {'average': 4.0, 'skipped': []}
''         -> ERROR: ValueError
'4, five'  -> ok: {'average': 4.0, 'skipped': ['five']}
'4,,5'     -> ok: {'average': 4.5, 'skipped': []}
'9, 1'     -> ok: {'average': 1.0, 'skipped': ['9']}
'4.5, 3'   -> ok: {'average': 3.0, 'skipped': ['4.5']}
```

In [ ]:
def average_from_text(text):
    """Average of comma-separated ratings, e.g. '4, 5, 3' -> 4.0"""
    ratings = [int(part) for part in text.split(",")]
    return round(sum(ratings) / len(ratings), 2)


def try_input(function, value):
    """Run function(value) and describe what happened, without crashing."""
    try:
        return f"ok: {function(value)}"
    except Exception as error:  # any error: we only want to record which one
        return f"ERROR: {type(error).__name__}"


cases = ["4, 5, 3", "", "4, five", "4,,5", "9, 1", "4.5, 3"]
for case in cases:
    shown = f"'{case}'"
    print(f"{shown:<10} -> {try_input(average_from_text, case)}")

In [ ]:
# your code here: write safe_average(text), then run the table again with it

Which case was the most dangerous: one that crashed, or one that gave a wrong answer **silently**? Why?

*Your answer:*

## Exercise 2: Redact contact details

Write `redact(text)` that replaces each word containing an e-mail address with `[email]` and each word with
**7 or more digits** with `[number]`. Strip `.,;:!?()` from a word before checking it.

Test it on the three texts below. The third comes from `reviews.csv` (review R33, fictional details).

Expected output:
```
Send it to [email] please
My number is [number] and my ID is S1042
Please call me at [number] or email [email] about my refund for this backpack.
```

In [ ]:
reviews = pd.read_csv(Path("../../../datasets/reviews.csv"))
texts = [
    "Send it to asha.k@example.com please",
    "My number is 0300-1234567 and my ID is S1042",
    reviews[reviews["review_id"] == "R33"]["review_text"].iloc[0],
]

# your code here

## Exercise 3: Are the numbers in the answer supported?

A FAQ assistant answered a question about late fees. The source section is given. Write
`unsupported_numbers(answer, source)` that returns the numbers in `answer` that do **not** appear anywhere in
`source` (compare as floats, so `10` and `10.00` count as the same number).

*Hint:* write a helper `numbers_in(text)` first (as in the demo).

Expected output:
```
[]
[1.0]
[50.0]
```
Then explain in one sentence why the third answer is flagged even though it is *correct*.

In [ ]:
source = "The late fee is 0.50 per book per day, up to a maximum of 10.00 per book."
answers = [
    "It costs 0.50 per day, and never more than 10 per book.",
    "The fee is 1.00 per book per day.",
    "It is 50 cents a day per book.",
]

# your code here

*Your answer:*

## Exercise 4: A counterfactual bias probe

A classmate built `tutoring_priority(student)`: a higher score means the student is offered tutoring sooner.
**Imagine you cannot read its code** (it could be a trained model or an LLM). Test it from the outside.

1. Write `probe(function, student, field, values)` that prints the score for each value of **one** field,
   keeping everything else the same (copy the dict with `student.copy()` before changing it).
2. Probe `program` (`"science"`, `"commerce"`, `"arts"`) and `previous_score` (`40`, `60`, `80`).
3. Which field changes the score although it should not matter for who needs help? Write one sentence.

Expected output:
```
program=science -> 50
program=commerce -> 50
program=arts -> 35
previous_score=40 -> 60
previous_score=60 -> 20
previous_score=80 -> -20
```

In [ ]:
def tutoring_priority(student):
    score = (60 - student["previous_score"]) * 2 + (80 - student["attendance_pct"])
    if student["program"] == "arts":
        score -= 15
    return score


asha = {"name": "Asha", "program": "science", "previous_score": 45, "attendance_pct": 60}

# your code here

*Your answer:*

## Exercise 5: Your README's Responsible AI section and demo script

Fill in both for **your own** capstone. Keep it short and specific. "It may be wrong" is not specific;
"answers about fees can be outdated because the FAQ file is from 2026" is.

**Responsible AI**
- Limitations (2–3, specific):
- Privacy: what is sent to the API (or "nothing: local model"), what is redacted, what users should not type:
- Bias: who is (not) represented in the data; what you probed; what you removed or kept, and why:
- When NOT to trust it:

**Demo script (5 minutes)**

| Time | What I show / say |
|------|-------------------|
| 0:00–0:30 | |
| 0:30–3:00 | normal case: … · edge case: … · failure handled: … |
| 3:00–4:00 | |
| 4:00–5:00 | |
| backup | |

## Exercise 6 (challenge): An automatic failure table

Write `check_cases(function, cases)` where `cases` is a list of `(input, expected)` pairs and `expected` is
either the expected return value or the **name** of the expected error, like `"ValueError"`. It prints `PASS`
or `FAIL` per case and returns the number of failures. Run it on your `safe_average` with the cases below.

Expected output:
```
PASS '4, 5, 3'
PASS ''
PASS '9, 1'
PASS 'x, y'
0 failure(s)
```

In [ ]:
expected_cases = [
    ("4, 5, 3", {"average": 4.0, "skipped": []}),
    ("", "ValueError"),
    ("9, 1", {"average": 1.0, "skipped": ["9"]}),
    ("x, y", "ValueError"),
]

# your code here